In [1]:
import os
import re
import json
import random
import warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import joblib
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: cuda


In [4]:
ID2LABEL = {
    0: 'sadness',
    1: 'joy',
    2: 'love',
    3: 'anger',
    4: 'fear',
    5: 'surprise',
}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(ID2LABEL)

TRAIN_PATH = 'train-00000-of-00001.parquet'
VAL_PATH   = 'validation-00000-of-00001.parquet'
TEST_PATH  = 'test-00000-of-00001.parquet'

for name, var in [('TRAIN_PATH', TRAIN_PATH), ('VAL_PATH', VAL_PATH), ('TEST_PATH', TEST_PATH)]:
    p = Path(var)
    if not p.exists():
        alt1 = Path('/content') / p.name
        if alt1.exists():
            globals()[name] = str(alt1)


MODEL_CHECKPOINT =  "distilroberta-base"
MAX_LENGTH = 160
OUTPUT_DIR = Path('clean_emotion_roberta')
OUTPUT_DIR.mkdir(exist_ok=True)
SHORT_WORD_THRESHOLD = 5

In [5]:
def load_split(path, split_name):
    df = pd.read_parquet(path).copy()
    required = {'text', 'label'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{split_name} is missing columns: {missing}')
    df = df[['text', 'label']].copy()
    df['text'] = df['text'].astype(str)
    df['label'] = df['label'].astype(int)
    df['split'] = split_name
    return df

train_df = load_split(TRAIN_PATH, 'train')
val_df   = load_split(VAL_PATH, 'validation')
test_df  = load_split(TEST_PATH, 'test')

for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    bad_labels = sorted(set(df['label']) - set(ID2LABEL.keys()))
    if bad_labels:
        raise ValueError(f'{name} has unexpected labels: {bad_labels}')
    print(name, df.shape)
    print(df['label'].map(ID2LABEL).value_counts().sort_index())
    print()

train (16000, 3)
label
anger       2159
fear        1937
joy         5362
love        1304
sadness     4666
surprise     572
Name: count, dtype: int64

validation (2000, 3)
label
anger       275
fear        212
joy         704
love        178
sadness     550
surprise     81
Name: count, dtype: int64

test (2000, 3)
label
anger       275
fear        224
joy         695
love        159
sadness     581
surprise     66
Name: count, dtype: int64



## 2. Leakage checks



In [6]:
def normalize_for_duplicate_check(text):
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

for df in [train_df, val_df, test_df]:
    df['dup_key'] = df['text'].map(normalize_for_duplicate_check)

val_keys = set(val_df['dup_key'])
test_keys = set(test_df['dup_key'])
train_before = len(train_df)

train_overlap_val = train_df['dup_key'].isin(val_keys).sum()
train_overlap_test = train_df['dup_key'].isin(test_keys).sum()
val_overlap_test = val_df['dup_key'].isin(test_keys).sum()

print('Train rows overlapping validation:', train_overlap_val)
print('Train rows overlapping test      :', train_overlap_test)
print('Validation rows overlapping test :', val_overlap_test)

train_df = train_df[~train_df['dup_key'].isin(val_keys | test_keys)].copy()
print(f'Removed from train: {train_before - len(train_df)}')
print('Clean train shape:', train_df.shape)

Train rows overlapping validation: 5
Train rows overlapping test      : 11
Validation rows overlapping test : 3
Removed from train: 16
Clean train shape: (15984, 4)


## 3. Minimal text normalization



In [7]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@[A-Za-z0-9_]+', ' ', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for df in [train_df, val_df, test_df]:
    df['clean_text'] = df['text'].map(clean_text)
    df['word_count'] = df['clean_text'].str.split().str.len().fillna(0).astype(int)

train_df = train_df[train_df['clean_text'].str.len() > 0].copy()
val_df   = val_df[val_df['clean_text'].str.len() > 0].copy()
test_df  = test_df[test_df['clean_text'].str.len() > 0].copy()

print('After cleaning:')
print('train:', train_df.shape, 'validation:', val_df.shape, 'test:', test_df.shape)

After cleaning:
train: (15984, 6) validation: (2000, 6) test: (2000, 6)


## 4. Train-only short-text augmentation



In [8]:
def make_short_training_augmentations(df, max_aug_per_row=2, min_words=3, max_words=8):
    rows = []
    rng = random.Random(SEED)
    for _, row in df.iterrows():
        words = str(row['clean_text']).split()
        if len(words) < max_words + 2:
            continue
        for _ in range(max_aug_per_row):
            span_len = rng.randint(min_words, min(max_words, len(words)))
            start = rng.randint(0, len(words) - span_len)
            snippet = ' '.join(words[start:start + span_len]).strip()
            if snippet and snippet.lower() != row['clean_text'].lower():
                rows.append({
                    'text': row['text'],
                    'clean_text': snippet,
                    'label': int(row['label']),
                    'split': 'train_augmented',
                    'word_count': len(snippet.split()),
                    'dup_key': normalize_for_duplicate_check(snippet),
                    'is_augmented': True,
                })
    return pd.DataFrame(rows)

train_df['is_augmented'] = False
aug_df = make_short_training_augmentations(train_df, max_aug_per_row=2)

if len(aug_df):
    aug_df = aug_df.drop_duplicates(subset=['dup_key', 'label'])

train_model_df = pd.concat([train_df, aug_df], ignore_index=True)
train_model_df = train_model_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print('Original train:', train_df.shape)
print('Augmented rows:', aug_df.shape)
print('Final training rows:', train_model_df.shape)
print('Short rows in final train:', (train_model_df['word_count'] <= SHORT_WORD_THRESHOLD).sum())

Original train: (15984, 7)
Augmented rows: (25161, 7)
Final training rows: (41145, 7)
Short rows in final train: 13278


## 5. Tokenization datasets

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)
class EmotionDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(map(int, labels))
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LENGTH,
            padding='max_length',
            return_tensors=None,
        )
        enc['labels'] = self.labels[idx]
        return {k: torch.tensor(v) for k, v in enc.items()}

train_dataset = EmotionDataset(train_model_df['clean_text'], train_model_df['label'])
val_dataset   = EmotionDataset(val_df['clean_text'], val_df['label'])
test_dataset  = EmotionDataset(test_df['clean_text'], test_df['label'])

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## 6. Train RoBERTa on train, select best checkpoint on validation



In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()
val_metrics = trainer.evaluate(val_dataset)
print(val_metrics)

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.861824,0.164452,0.934500,0.909765,0.934084
2,0.806976,0.157172,0.931500,0.908680,0.931378
3,0.703553,0.146335,0.932500,0.910600,0.933249
4,0.670833,0.158442,0.933500,0.913671,0.933858


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 7. Train a character n-gram fallback on train only



In [ ]:
char_ngram_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2, 5),
        min_df=1,
        sublinear_tf=True,
    )),
    ('clf', LogisticRegression(
        max_iter=3000,
        class_weight='balanced',
        C=4.0,
        n_jobs=-1,
        random_state=SEED,
    ))
])

char_ngram_model.fit(train_model_df['clean_text'], train_model_df['label'])
val_char_probs = char_ngram_model.predict_proba(val_df['clean_text'])
val_char_pred = val_char_probs.argmax(axis=1)
print('Char ngram validation accuracy:', accuracy_score(val_df['label'], val_char_pred))
print('Char ngram validation macro-F1:', f1_score(val_df['label'], val_char_pred, average='macro'))

## 8. Tune ensemble on validation only


In [ ]:
def softmax_np(x):
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

val_roberta_output = trainer.predict(val_dataset)
val_roberta_probs = softmax_np(val_roberta_output.predictions)
y_val = val_df['label'].values

best = None
for w_roberta in np.linspace(0.0, 1.0, 21):
    probs = w_roberta * val_roberta_probs + (1.0 - w_roberta) * val_char_probs
    preds = probs.argmax(axis=1)
    macro = f1_score(y_val, preds, average='macro')
    acc = accuracy_score(y_val, preds)
    candidate = {'w_roberta': float(w_roberta), 'macro_f1': float(macro), 'accuracy': float(acc)}
    if best is None or candidate['macro_f1'] > best['macro_f1']:
        best = candidate

print('Best validation ensemble:', best)
ENSEMBLE_W_ROBERTA = best['w_roberta']

## 9. Final test evaluation — run once



In [ ]:
test_roberta_output = trainer.predict(test_dataset)
test_roberta_probs = softmax_np(test_roberta_output.predictions)
test_char_probs = char_ngram_model.predict_proba(test_df['clean_text'])

test_final_probs = ENSEMBLE_W_ROBERTA * test_roberta_probs + (1.0 - ENSEMBLE_W_ROBERTA) * test_char_probs
test_preds = test_final_probs.argmax(axis=1)
y_test = test_df['label'].values

print('FINAL TEST ACCURACY:', accuracy_score(y_test, test_preds))
print('FINAL TEST MACRO-F1:', f1_score(y_test, test_preds, average='macro'))
print('FINAL TEST WEIGHTED-F1:', f1_score(y_test, test_preds, average='weighted'))
print('Classification report:')
print(classification_report(y_test, test_preds, target_names=[ID2LABEL[i] for i in range(NUM_LABELS)]))

FINAL TEST ACCURACY: 0.92
FINAL TEST MACRO-F1: 0.8768253950248428
FINAL TEST WEIGHTED-F1: 0.9207950373306882
Classification report:
              precision    recall  f1-score   support

     sadness       0.96      0.96      0.96       581
         joy       0.96      0.93      0.95       695
        love       0.78      0.87      0.82       159
       anger       0.92      0.91      0.91       275
        fear       0.88      0.88      0.88       224
    surprise       0.72      0.77      0.74        66

    accuracy                           0.92      2000
   macro avg       0.87      0.89      0.88      2000
weighted avg       0.92      0.92      0.92      2000



## 10. Short-text vs long-text evaluation



In [ ]:
def evaluate_bucket(name, mask):
    idx = np.where(mask)[0]
    if len(idx) == 0:
        print(f'{name}: no samples')
        return
    print(name, '| samples:', len(idx))
    print('Accuracy:', accuracy_score(y_test[idx], test_preds[idx]))
    print('Macro-F1:', f1_score(y_test[idx], test_preds[idx], average='macro'))
    print(classification_report(
        y_test[idx],
        test_preds[idx],
        labels=list(range(NUM_LABELS)),
        target_names=[ID2LABEL[i] for i in range(NUM_LABELS)],
        zero_division=0,
    ))

wc = test_df['word_count'].values
evaluate_bucket('Very short text: <= 2 words', wc <= 2)
evaluate_bucket('Short text: <= 5 words', wc <= 5)
evaluate_bucket('Medium text: 6-20 words', (wc > 5) & (wc <= 20))
evaluate_bucket('Long text: > 20 words', wc > 20)

Very short text: <= 2 words: no samples
Short text: <= 5 words | samples: 92
Accuracy: 0.9456521739130435
Macro-F1: 0.9070450480661747
              precision    recall  f1-score   support

     sadness       0.97      0.94      0.96        36
         joy       1.00      0.91      0.95        33
        love       0.75      1.00      0.86         3
       anger       0.78      1.00      0.88         7
        fear       1.00      1.00      1.00        11
    surprise       0.67      1.00      0.80         2

    accuracy                           0.95        92
   macro avg       0.86      0.98      0.91        92
weighted avg       0.96      0.95      0.95        92

Medium text: 6-20 words | samples: 1138
Accuracy: 0.9147627416520211
Macro-F1: 0.8640526386254299
              precision    recall  f1-score   support

     sadness       0.95      0.97      0.96       316
         joy       0.96      0.93      0.95       396
        love       0.78      0.88      0.83        90
       

## 11. Error analysis

In [ ]:
error_df = test_df[['text', 'clean_text', 'label', 'word_count']].copy()
error_df['true_emotion'] = error_df['label'].map(ID2LABEL)
error_df['pred_label'] = test_preds
error_df['pred_emotion'] = error_df['pred_label'].map(ID2LABEL)
error_df['confidence'] = test_final_probs.max(axis=1)
error_df['correct'] = error_df['label'] == error_df['pred_label']

print('Number of errors:', (~error_df['correct']).sum())
error_df[~error_df['correct']].sort_values('confidence', ascending=False).head(30)

Number of errors: 160


,text,clean_text,label,word_count,true_emotion,pred_label,pred_emotion,confidence,correct
1382,i cannot even begin to express in words the de...,i cannot even begin to express in words the de...,5,27,surprise,0,sadness,0.984891,False
1882,i feel watching him grow into a self assured l...,i feel watching him grow into a self assured l...,1,12,joy,2,love,0.983436,False
542,i also feel i have accepted my dark side and a...,i also feel i have accepted my dark side and a...,1,20,joy,2,love,0.969432,False
1627,i feel i can only hope im not alone in these t...,i feel i can only hope im not alone in these t...,0,48,sadness,1,joy,0.966415,False
816,whenever i put myself in others shoes and try ...,whenever i put myself in others shoes and try ...,3,14,anger,1,joy,0.963773,False
1206,ive come to feel about a supporting character ...,ive come to feel about a supporting character ...,1,17,joy,2,love,0.962354,False
590,i was supposed to feel sympathy for emma im af...,i was supposed to feel sympathy for emma im af...,4,12,fear,0,sadness,0.958228,False
20,im not sure the feeling of loss will ever go a...,im not sure the feeling of loss will ever go a...,0,42,sadness,2,love,0.957943,False
1928,i feel inside cause life is like a game someti...,i feel inside cause life is like a game someti...,4,41,fear,0,sadness,0.951437,False
1959,i check you when you re sleeping feel your nos...,i check you when you re sleeping feel your nos...,1,22,joy,2,love,0.944900,False


## 12. Save clean model artifacts



In [ ]:
FINAL_DIR = OUTPUT_DIR / 'final_model'
FINAL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_DIR / 'roberta'))
tokenizer.save_pretrained(str(FINAL_DIR / 'roberta'))
joblib.dump(char_ngram_model, FINAL_DIR / 'char_ngram_model.pkl')

config = {
    'id2label': {str(k): v for k, v in ID2LABEL.items()},
    'label2id': LABEL2ID,
    'model_checkpoint': MODEL_CHECKPOINT,
    'max_length': MAX_LENGTH,
    'short_word_threshold_for_reporting': SHORT_WORD_THRESHOLD,
    'ensemble_w_roberta': ENSEMBLE_W_ROBERTA,
    'cleaning': 'neutral_url_mention_whitespace_cleaning_only',
    'no_label_keyword_injection': True,
    'test_accuracy': float(accuracy_score(y_test, test_preds)),
    'test_macro_f1': float(f1_score(y_test, test_preds, average='macro')),
}
with open(FINAL_DIR / 'inference_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print('Saved clean artifacts to:', FINAL_DIR)
for p in FINAL_DIR.rglob('*'):
    print(' -', p)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved clean artifacts to: clean_emotion_roberta/final_model
 - clean_emotion_roberta/final_model/char_ngram_model.pkl
 - clean_emotion_roberta/final_model/inference_config.json
 - clean_emotion_roberta/final_model/roberta
 - clean_emotion_roberta/final_model/roberta/tokenizer_config.json
 - clean_emotion_roberta/final_model/roberta/config.json
 - clean_emotion_roberta/final_model/roberta/tokenizer.json
 - clean_emotion_roberta/final_model/roberta/model.safetensors
 - clean_emotion_roberta/final_model/roberta/training_args.bin


## 13. Clean inference function


In [ ]:
import torch
import joblib
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pathlib import Path

ARTIFACT_DIR = Path('clean_emotion_roberta/final_model')
ROBERTA_DIR = ARTIFACT_DIR / 'roberta'

with open(ARTIFACT_DIR / 'inference_config.json', 'r', encoding='utf-8') as f:
    infer_cfg = json.load(f)

infer_id2label = {int(k): v for k, v in infer_cfg['id2label'].items()}
infer_tokenizer = AutoTokenizer.from_pretrained(str(ROBERTA_DIR))
infer_model = AutoModelForSequenceClassification.from_pretrained(str(ROBERTA_DIR))
infer_model.eval()
infer_char_model = joblib.load(ARTIFACT_DIR / 'char_ngram_model.pkl')
W_ROBERTA = float(infer_cfg['ensemble_w_roberta'])


def predict_emotion_clean(text: str):
    cleaned = clean_text(text)
    if not cleaned:
        return {
            'emotion': 'unknown',
            'confidence': 0.0,
            'probabilities': {v: 0.0 for v in infer_id2label.values()},
            'clean_text': cleaned,
        }

    enc = infer_tokenizer(
        cleaned,
        return_tensors='pt',
        truncation=True,
        max_length=infer_cfg['max_length'],
        padding=True,
    )
    with torch.no_grad():
        logits = infer_model(**enc).logits.cpu().numpy()
    roberta_probs = softmax_np(logits)[0]
    char_probs = infer_char_model.predict_proba([cleaned])[0]
    probs = W_ROBERTA * roberta_probs + (1.0 - W_ROBERTA) * char_probs

    pred_id = int(np.argmax(probs))
    return {
        'emotion': infer_id2label[pred_id],
        'confidence': round(float(probs[pred_id]), 4),
        'probabilities': {infer_id2label[i]: round(float(probs[i]), 4) for i in range(len(probs))},
        'clean_text': cleaned,
    }

# Examples
for txt in ['I am terrified', 'no response', 'I feel happy today', 'I feel empty and exhausted every night']:
    print(txt, '=>', predict_emotion_clean(txt))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

I am terrified => {'emotion': 'fear', 'confidence': 0.9752, 'probabilities': {'sadness': 0.0063, 'joy': 0.0139, 'love': 0.0026, 'anger': 0.0016, 'fear': 0.9752, 'surprise': 0.0006}, 'clean_text': 'I am terrified'}
no response => {'emotion': 'anger', 'confidence': 0.476, 'probabilities': {'sadness': 0.2879, 'joy': 0.0407, 'love': 0.0357, 'anger': 0.476, 'fear': 0.126, 'surprise': 0.0336}, 'clean_text': 'no response'}
I feel happy today => {'emotion': 'joy', 'confidence': 0.9799, 'probabilities': {'sadness': 0.0156, 'joy': 0.9799, 'love': 0.0009, 'anger': 0.0026, 'fear': 0.0005, 'surprise': 0.0006}, 'clean_text': 'I feel happy today'}
I feel empty and exhausted every night => {'emotion': 'sadness', 'confidence': 0.9984, 'probabilities': {'sadness': 0.9984, 'joy': 0.0005, 'love': 0.0001, 'anger': 0.0004, 'fear': 0.0002, 'surprise': 0.0003}, 'clean_text': 'I feel empty and exhausted every night'}


## 14. Optional Hugging Face upload



In [ ]:
from huggingface_hub import login, create_repo, upload_folder
login()
HF_REPO_ID = 'MariamMohsen112/clean-emotion-roberta'
create_repo(HF_REPO_ID, repo_type='model', private=False, exist_ok=True)
upload_folder(
    repo_id=HF_REPO_ID,
    repo_type='model',
    folder_path=str(FINAL_DIR / 'roberta'),
    commit_message='Upload clean emotion RoBERTa model'
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...roberta/model.safetensors:   2%|1         | 5.50MB /  329MB            

  ...roberta/training_args.bin:   6%|6         |   334B / 5.20kB            

CommitInfo(commit_url='https://huggingface.co/MariamMohsen112/clean-emotion-roberta/commit/345c41200f9af8b6ed2fc43ddadc483b63580a30', commit_message='Upload clean emotion RoBERTa model', commit_description='', oid='345c41200f9af8b6ed2fc43ddadc483b63580a30', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MariamMohsen112/clean-emotion-roberta', endpoint='https://huggingface.co', repo_type='model', repo_id='MariamMohsen112/clean-emotion-roberta'), pr_revision=None, pr_num=None)